<a href="https://colab.research.google.com/github/simonburner/hands-on-LLM/blob/main/ch2/wordpiece_tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
corpus = [
    "I want to learn how the WordPiece tokenization algorithm works "
    "This project follows the process how the WordPiece tokenization algorithm works "
    "Let's learn how they work"
]

In [ ]:
from transformers import AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [ ]:
from collections import defaultdict

In [ ]:
word_freqs = defaultdict(int)

for text in corpus:
  words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
  new_words = [word for word, offset in words_with_offsets]
  for word in new_words:
    word_freqs[word] += 1

print(word_freqs)

defaultdict(<class 'int'>, {'I': 1, 'want': 1, 'to': 1, 'learn': 2, 'how': 3, 'the': 3, 'WordPiece': 2, 'tokenization': 2, 'algorithm': 2, 'works': 2, 'This': 1, 'project': 1, 'follows': 1, 'process': 1, 'Let': 1, "'": 1, 's': 1, 'they': 1, 'work': 1})


In [ ]:
alphabet = []

for word in word_freqs.keys():
  if word[0] not in alphabet:
    alphabet.append(word[0])
  for letter in word[1:]:
    if f"##{letter}" not in alphabet:
      alphabet.append(f"##{letter}")

alphabet.sort()

print(alphabet)

['##P', '##a', '##c', '##d', '##e', '##g', '##h', '##i', '##j', '##k', '##l', '##m', '##n', '##o', '##r', '##s', '##t', '##w', '##y', '##z', "'", 'I', 'L', 'T', 'W', 'a', 'f', 'h', 'l', 'p', 's', 't', 'w']


In [ ]:
vocab = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + alphabet.copy()

In [ ]:
splits = {
    word: [c if i == 0 else f"##{c}" for i, c in enumerate(word)]
    for word in word_freqs.keys()
}

In [ ]:
def compute_pair_scores(splits):
  letter_freqs = defaultdict(int)
  pair_freqs = defaultdict(int)
  for word, freq in word_freqs.items():
    split = splits[word]
    if len(split) == 1:
      letter_freqs[split[0]] += freq
      continue

    for i in range(len(split) - 1):
      pair = (split[i], split[i + 1])
      letter_freqs[split[i]] += freq
      pair_freqs[pair] += freq
    letter_freqs[split[-1]] += freq

    scores = {
        pair: freq / (letter_freqs[pair[0]] * letter_freqs[pair[1]])
        for pair, freq in pair_freqs.items()
    }

    return scores

In [ ]:
pair_scores = compute_pair_scores(splits)

for i, key in enumerate(pair_scores.keys()):
  print(f"{key}: {pair_scores[key]}")
  if i >= 5:
    break

('w', '##a'): 1.0
('##a', '##n'): 1.0
('##n', '##t'): 1.0


In [ ]:
best_pair = ""
max_score = None

for pair, score in pair_scores.items():
  if max_score is None or max_score < score:
    best_pair = pair
    max_score = score

print(best_pair, max_score)

('w', '##a') 1.0


In [ ]:
vocab.append("wa")

In [ ]:
def merge_pair(a, b, splits):
  for word in word_freqs:
    split = splits[word]
    if len(split) == 1:
      continue

    i = 0
    while i < len(split) - 1:
      if split[i] == a and split[i + 1] == b:
        merge = a + b[2:] if b.startswith("##") else a + b
        split = split[:i] + [merge] + split[i + 2 :]
      else:
        i += 1
    splits[word] = split

  return splits

In [ ]:
splits = merge_pair("w", "##a", splits)
splits["want"]

['wa', '##n', '##t']

In [ ]:
vocab_size = 70
while len(vocab) < vocab_size:
  scores = compute_pair_scores(splits)
  best_pair = ""
  max_score = None
  for pair, score in scores.items():
    if max_score is None or max_score < score:
      best_pair = pair
      max_score = score
  splits = merge_pair(*best_pair, splits)
  new_token = (
      best_pair[0] + best_pair[1][2:]
      if best_pair[1].startswith("##")
      else best_pair[0] + best_pair[1]
  )
  vocab.append(new_token)

In [ ]:
print(vocab)

['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', '##P', '##a', '##c', '##d', '##e', '##g', '##h', '##i', '##j', '##k', '##l', '##m', '##n', '##o', '##r', '##s', '##t', '##w', '##y', '##z', "'", 'I', 'L', 'T', 'W', 'a', 'f', 'h', 'l', 'p', 's', 't', 'w', 'wa', 'wan', 'want', 'to', 'le', 'lea', 'lear', 'learn', 'ho', 'how', 'th', 'the', 'Wo', 'Wor', 'Word', 'WordP', 'WordPi', 'WordPie', 'WordPiec', 'WordPiece', '##ke', '##za', '##zat', 'toke', 'token', '##on', 'tokeni', 'tokenizat', 'tokenizati', 'tokenization', 'al', 'alg']
